# Billing cost model

Shows the work behind [`docs/BILLING_COST_MODEL.md`](../docs/BILLING_COST_MODEL.md):
every per-tier cap and price in the product traced back to a vendor rate times a
usage assumption.

**Three inputs, all editable:**

| File | What it holds |
| --- | --- |
| [`pricing.json`](pricing.json) | vendor unit prices, keyed by model and voice family |
| [`assumptions.json`](assumptions.json) | how much of each service a lecture and its audience consume |
| [`../config/plans.json`](../config/plans.json) | the caps actually shipped — what we are evaluating |

Change a number, `Run All`, read the new answer. Nothing is hard-coded in the
notebook; stdlib only, so it runs in any kernel.

> Prices were verified 2026-07-31 against the vendors' own pricing pages —
> §1 lists every source URL. The usage assumptions are educated guesses; the
> ones marked `weak` in `assumptions.json` are what a semester of pilot data
> should replace.


## 1. Load the inputs


In [37]:
import json, os
from IPython.display import Markdown, display

ROOT = os.path.abspath('..')
P = json.load(open('pricing.json'))
A = json.load(open('assumptions.json'))
PLANS = json.load(open(os.path.join(ROOT, 'config', 'plans.json')))

def table(headers, rows, title=None):
    """Renders a markdown table so output reads like the doc it explains."""
    out = (f'**{title}**\n\n' if title else '')
    out += '| ' + ' | '.join(headers) + ' |\n'
    out += '|' + '|'.join(['---'] * len(headers)) + '|\n'
    for r in rows:
        out += '| ' + ' | '.join(str(c) for c in r) + ' |\n'
    display(Markdown(out))

def usd(x, dp=2):
    return f'${x:,.{dp}f}'

print(f"pricing asOf {P['asOf']} | tiers {list(PLANS)} | default model {P['ai']['defaultModel']}")


pricing asOf 2026-07-31 | tiers ['free', 'fresh', 'pro', 'max'] | default model gemini-3.1-flash-lite-preview


### Where these prices come from

Every rate in `pricing.json` traces to a vendor pricing page. Rendered from the
data rather than typed here, so adding a price and its source keeps them
together — and `verified` says when a human last read the page, which is the
number that goes stale.


In [38]:
rows = [[f"[{v['service']}]({v['url']})", v['verified'], v.get('note', '')]
        for v in P['_sources'].values()]
table(['Service', 'Verified', 'Gotchas worth remembering'], rows,
      'Pricing sources')
print('Raw URLs, for when the markdown links are not clickable:')
for k, v in P['_sources'].items():
    print(f"  {k:12s} {v['url']}")


**Pricing sources**

| Service | Verified | Gotchas worth remembering |
|---|---|---|
| [Gemini API (text, embeddings, image generation)](https://ai.google.dev/gemini-api/docs/pricing) | 2026-07-31 |  |
| [Google Cloud Speech-to-Text](https://cloud.google.com/speech-to-text/pricing) | 2026-07-31 | V2 table. One Recognition SKU covers streaming AND standard batch; Dynamic Batch is a separate opt-in SKU. No free tier on V2 - the 60 free minutes are V1 only. Requests round up to 1 second. |
| [Google Cloud Text-to-Speech](https://cloud.google.com/text-to-speech/pricing) | 2026-07-31 | Per character of input, including SSML tags except <mark>. Each voice family has its own monthly free allowance. Gemini-TTS models are priced per token instead. |
| [Google Cloud Translation](https://cloud.google.com/translate/pricing) | 2026-07-31 | First 500,000 chars/month free per account, not per user. |
| [DigitalOcean Spaces (object storage + egress)](https://www.digitalocean.com/pricing/spaces-object-storage) | 2026-07-31 |  |
| [DigitalOcean App Platform (fixed cost)](https://www.digitalocean.com/pricing/app-platform) | 2026-07-31 |  |
| [MongoDB Atlas (fixed cost)](https://www.mongodb.com/pricing) | 2026-07-31 | M10 dedicated; backups are billed by snapshot storage and grow with the data. |
| [Stripe payments + Billing](https://stripe.com/pricing) | 2026-07-31 | 2.9% + $0.30 per successful charge, plus 0.7% of volume for Billing on pay-as-you-go. |


Raw URLs, for when the markdown links are not clickable:
  gemini       https://ai.google.dev/gemini-api/docs/pricing
  stt          https://cloud.google.com/speech-to-text/pricing
  tts          https://cloud.google.com/text-to-speech/pricing
  translation  https://cloud.google.com/translate/pricing
  spaces       https://www.digitalocean.com/pricing/spaces-object-storage
  appPlatform  https://www.digitalocean.com/pricing/app-platform
  atlas        https://www.mongodb.com/pricing
  stripe       https://stripe.com/pricing


### Drift check

`pricing.json` is the modelling copy so you can experiment freely. The server
reads [`../config/service-prices.json`](../config/service-prices.json). If the two
diverge, the model is describing a product we do not ship — so check.


In [39]:
live = json.load(open(os.path.join(ROOT, 'config', 'service-prices.json')))

def flat(d, prefix=''):
    out = {}
    for k, v in d.items():
        if k.startswith('_'):
            continue
        key = f'{prefix}{k}'
        out.update(flat(v, key + '.') if isinstance(v, dict) else {key: v})
    return out

a, b = flat(P), flat(live)
drift = [(k, a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)]
if drift:
    table(['key', 'pricing.json', 'config/service-prices.json'], drift,
          'DRIFT — the model and the server disagree')
else:
    print('No drift: modelling prices match the shipped config.')


No drift: modelling prices match the shipped config.


## 2. Unit prices

Pulled by name, so switching `GEMINI_MODEL` or a narration voice family in the
config changes the model too.


In [40]:
model = P['ai']['models'][P['ai']['defaultModel']]
TOK_IN   = model['inputPerMillionTokens'] / 1e6
TOK_OUT  = model['outputPerMillionTokens'] / 1e6
EMBED    = list(P['ai']['embeddingModels'].values())[0]['inputPerMillionTokens'] / 1e6
IMAGE    = list(P['ai']['imageModels'].values())[0]['perImage']

# One SKU covers streaming AND standard batch on STT V2 — diarization included.
STT      = P['stt']['recognitionPerMinute']
TTS_STD  = P['tts']['voiceFamilies'][P['tts']['defaultStandardFamily']]['perMillionChars'] / 1e6
TTS_PREM = P['tts']['voiceFamilies'][P['tts']['defaultPremiumFamily']]['perMillionChars'] / 1e6
TRANSLATE = P['translation']['perMillionChars'] / 1e6
GIB_MONTH, EGRESS = P['storage']['perGibMonth'], P['storage']['egressPerGib']
PAY_RATE  = P['payments']['rate'] + P['payments']['billingRate']
PAY_FIXED = P['payments']['perTransaction']

src = lambda key: f"[source]({P['_sources'][key]['url']})"
table(['Unit', 'Price', 'Source'], [
    [f"{P['ai']['defaultModel']} input",  f'{usd(TOK_IN*1e6)} / 1M tokens', src('gemini')],
    [f"{P['ai']['defaultModel']} output", f'{usd(TOK_OUT*1e6)} / 1M tokens', src('gemini')],
    ['STT recognition (streaming + batch)', f'{usd(STT,3)} / min', src('stt')],
    ['STT dynamic batch (24h turnaround)', f"{usd(P['stt']['dynamicBatchPerMinute'],3)} / min", src('stt')],
    [f"TTS {P['tts']['defaultStandardFamily']}",  f'{usd(TTS_STD*1e6)} / 1M chars', src('tts')],
    [f"TTS {P['tts']['defaultPremiumFamily']}", f'{usd(TTS_PREM*1e6)} / 1M chars', src('tts')],
    ['Translation', f'{usd(TRANSLATE*1e6)} / 1M chars', src('translation')],
    ['Storage / egress', f'{usd(GIB_MONTH,3)} per GiB-month / {usd(EGRESS,3)} per GiB', src('spaces')],
    ['Payments', f'{PAY_RATE*100:.1f}% + {usd(PAY_FIXED)}', src('stripe')],
], 'Unit prices in play')


**Unit prices in play**

| Unit | Price | Source |
|---|---|---|
| gemini-3.1-flash-lite-preview input | $0.25 / 1M tokens | [source](https://ai.google.dev/gemini-api/docs/pricing) |
| gemini-3.1-flash-lite-preview output | $1.50 / 1M tokens | [source](https://ai.google.dev/gemini-api/docs/pricing) |
| STT recognition (streaming + batch) | $0.016 / min | [source](https://cloud.google.com/speech-to-text/pricing) |
| STT dynamic batch (24h turnaround) | $0.003 / min | [source](https://cloud.google.com/speech-to-text/pricing) |
| TTS neural2 | $16.00 / 1M chars | [source](https://cloud.google.com/text-to-speech/pricing) |
| TTS chirp3-hd | $30.00 / 1M chars | [source](https://cloud.google.com/text-to-speech/pricing) |
| Translation | $20.00 / 1M chars | [source](https://cloud.google.com/translate/pricing) |
| Storage / egress | $0.020 per GiB-month / $0.010 per GiB | [source](https://www.digitalocean.com/pricing/spaces-object-storage) |
| Payments | 3.6% + $0.30 | [source](https://stripe.com/pricing) |


## 3. One lecture

Everything scales from **lecture duration**. Change
`assumptions.lecture.durationMinutes` and every figure below moves with it.


In [41]:
def per_lecture(duration=None):
    L, X, R, AU = A['lecture'], A['perLectureExtras'], A['revision'], A['audience']
    d = duration or L['durationMinutes']
    scale = d / L['durationMinutes']
    slides = L['slidesPerLecture'] * scale
    calls  = L['phrasesPerMinute'] * d

    generation = calls * (L['inputTokensPerCall'] * TOK_IN + L['outputTokensPerCall'] * TOK_OUT)
    rerank = slides * (X['rerankInputTokensPerSlide'] * TOK_IN + X['rerankOutputTokensPerSlide'] * TOK_OUT)
    quiz = X['quizInputTokens'] * TOK_IN + X['quizOutputTokens'] * TOK_OUT
    embeddings = X['embeddingTokens'] * scale * EMBED
    stt = d * L['sttMinutesPerLectureMinute'] * STT

    refined = slides * R['refinedSlideShare']
    refine = refined * (R['refineInputTokensPerSlide'] * TOK_IN + R['refineOutputTokensPerSlide'] * TOK_OUT)
    narrate = refined * (R['narrateInputTokensPerSlide'] * TOK_IN + R['narrateOutputTokensPerSlide'] * TOK_OUT)
    narration_chars = slides * L['narrationCharsPerSlide']
    tts = narration_chars * TTS_STD
    resynth = R['resynthesizedSlides'] * L['narrationCharsPerSlide'] * TTS_STD
    retranscribe = R['retranscribeMinutes'] * STT   # streaming-priced, not batch
    diarize = R['diarizeShareOfLectures'] * d * STT

    locales = AU['localesPerDeck']
    translate = locales * slides * L['slideTextChars'] * TRANSLATE
    translated_tts = locales * narration_chars * TTS_STD
    playbacks = AU['studentsPerSection'] * AU['deckOpenRate'] * AU['playbacksPerViewingStudent']
    egress = playbacks * (AU['imageMbPerPlayback'] + AU['narrationAudioMbPerPlayback']) / 1024 * EGRESS
    audio_mb = d * L['captureSampleRateHz'] * L['captureBytesPerSample'] * 60 / 1e6
    storage = audio_mb / 1024 * GIB_MONTH

    lines = {
        'Slide generation': generation, 'Image re-rank': rerank,
        'Quiz generation': quiz, 'Embeddings': embeddings,
        'Cloud STT (live)': stt, 'Refine + narrate passes': refine + narrate,
        'TTS narration': tts, 'TTS re-synthesis': resynth,
        'Cloud STT (re-transcribe)': retranscribe, 'Diarization': diarize,
        'Translation (audience)': translate,
        'Translated narration (audience)': translated_tts,
        'Playback egress (audience)': egress, 'Audio storage': storage,
    }
    return {'lines': lines, 'slides': slides, 'audioMb': audio_mb,
            'aiTokens': calls * (L['inputTokensPerCall'] + L['outputTokensPerCall']),
            'narrationChars': narration_chars, 'playbacks': playbacks,
            # Browser capture has no cloud STT — and no retained audio, so no
            # diarization and no re-transcription either.
            'cloudOnly': stt + retranscribe + diarize}

r = per_lecture()
total = sum(r['lines'].values())
table(['Line', 'Cost'], [[k, usd(v, 3)] for k, v in r['lines'].items()],
      f"One {A['lecture']['durationMinutes']}-minute lecture")
print(f"{r['slides']:.0f} slides | {r['aiTokens']/1e6:.2f}M tokens | "
      f"{r['narrationChars']:,.0f} narration chars | {r['audioMb']:.0f} MB audio | "
      f"{r['playbacks']:.0f} playbacks")
print(f"TOTAL {usd(total)} with cloud capture | {usd(total - r['cloudOnly'])} on browser capture")


**One 75-minute lecture**

| Line | Cost |
|---|---|
| Slide generation | $0.861 |
| Image re-rank | $0.030 |
| Quiz generation | $0.006 |
| Embeddings | $0.003 |
| Cloud STT (live) | $1.200 |
| Refine + narrate passes | $0.028 |
| TTS narration | $0.432 |
| TTS re-synthesis | $0.048 |
| Cloud STT (re-transcribe) | $0.048 |
| Diarization | $0.360 |
| Translation (audience) | $0.180 |
| Translated narration (audience) | $0.216 |
| Playback egress (audience) | $0.003 |
| Audio storage | $0.004 |


45 slides | 2.26M tokens | 27,000 narration chars | 216 MB audio | 20 playbacks
TOTAL $3.42 with cloud capture | $1.81 on browser capture


## 4. Per 100 lectures, by service

The fastest read: which services actually cost money, and how much of it the
audience drives.


In [42]:
L = r['lines']
svc = {
    'Cloud STT': L['Cloud STT (live)'] + L['Cloud STT (re-transcribe)'],
    'Gemini (all LLM)': L['Slide generation'] + L['Image re-rank'] + L['Quiz generation']
                        + L['Embeddings'] + L['Refine + narrate passes'],
    'TTS': L['TTS narration'] + L['TTS re-synthesis'] + L['Translated narration (audience)'],
    'Diarization': L['Diarization'],
    'Translation': L['Translation (audience)'],
    'Storage + egress': L['Audio storage'] + L['Playback egress (audience)'],
}
svc = {k: v * 100 for k, v in svc.items()}
tot = sum(svc.values())
student = (L['Translation (audience)'] + L['Translated narration (audience)']
           + L['Playback egress (audience)']) * 100

table(['Service', 'Cost / 100 lectures', 'Share'],
      [[k, usd(v), f'{v/tot*100:.1f}%'] for k, v in sorted(svc.items(), key=lambda x: -x[1])]
      + [['**Total**', f'**{usd(tot)}**', '']],
      'Per 100 lectures')
print(f"Browser-capture tiers: {usd(tot - r['cloudOnly']*100)} "
      f"({(r['cloudOnly']*100)/tot*100:.0f}% of cost removed with one config value)")
print(f"Instructor-driven {usd(tot-student)} ({(tot-student)/tot*100:.0f}%) | "
      f"student-driven {usd(student)} ({student/tot*100:.0f}%)")


**Per 100 lectures**

| Service | Cost / 100 lectures | Share |
|---|---|---|
| Cloud STT | $124.80 | 36.5% |
| Gemini (all LLM) | $92.83 | 27.1% |
| TTS | $69.60 | 20.4% |
| Diarization | $36.00 | 10.5% |
| Translation | $18.00 | 5.3% |
| Storage + egress | $0.70 | 0.2% |
| **Total** | **$341.93** |  |


Browser-capture tiers: $181.13 (47% of cost removed with one config value)
Instructor-driven $302.05 (88%) | student-driven $39.87 (12%)


## 5. What each tier costs

Reads the **shipped** caps from `config/plans.json`, so editing a cap there and
rerunning shows its price impact immediately.

An unlimited cap (`null`) contributes nothing — an unbounded cap has no worst
case, which is exactly why no tier has one.

Four utilisation levels, since subscribers are not all the same shape:

- **Light** — subscribed for the capability rather than the volume: a few
  lectures a month, little revision. Common, and the most profitable.
- **Expected** — the planning case, half the caps.
- **Heavy** — a power user running close to the limits.
- **At caps** — everything consumed. Rare, but it is the number the price has
  to survive.


In [43]:
BLEND_INPUT = 0.93  # ~93% of our tokens are prompt, not completion
AI_BLENDED = (BLEND_INPUT * model['inputPerMillionTokens']
              + (1 - BLEND_INPUT) * model['outputPerMillionTokens']) / 1e6
SLIDE_TEXT_PER_DECK = A['lecture']['slidesPerLecture'] * A['lecture']['slideTextChars']

def worst_case(caps):
    v = lambda k: caps.get(k) or 0   # None (unlimited) contributes nothing
    return {
        'aiTokens': v('aiTokens') * AI_BLENDED,
        'sttMinutes': v('sttMinutes') * STT,
        'diarizationMinutes': v('diarizationMinutes') * STT,
        'ttsCharacters': v('ttsCharacters') * TTS_STD,
        'ttsPremiumCharacters': v('ttsPremiumCharacters') * TTS_PREM,
        'aiImages': v('aiImages') * IMAGE,
        'translationCharacters': v('translationCharacters') * TRANSLATE,
        'audienceTtsCharacters': v('audienceTtsCharacters') * TTS_STD,
        'audienceLocales': v('audienceLocales') * SLIDE_TEXT_PER_DECK * TRANSLATE,
        'audioStorageMb': v('audioStorageMb') / 1024 * GIB_MONTH,
    }

floor_price = lambda cost: (cost + PAY_FIXED) / (1 - PAY_RATE)
TG = A['targets']
rows, breakdown = [], {}
for tier, plan in PLANS.items():
    parts = worst_case(plan['caps'])
    breakdown[tier] = parts
    wc = sum(parts.values())
    price = A['tiers'][tier]['priceUsd']
    rows.append([tier, A['tiers'][tier]['lecturesPerMonth'],
                 usd(wc * TG['lightUtilisation']), usd(wc * TG['expectedUtilisation']),
                 usd(wc * TG['heavyUtilisation']), usd(wc), usd(floor_price(wc)),
                 usd(price), f'{wc/price*100:.0f}%' if price else '—',
                 '✅' if not price or wc/price <= TG['worstCaseShareOfPrice'] + 0.15 else '⚠️'])
table(['Tier', 'Lectures/mo', f"Light ({TG['lightUtilisation']:.0%})",
       f"Expected ({TG['expectedUtilisation']:.0%})", f"Heavy ({TG['heavyUtilisation']:.0%})",
       'At caps', 'Price floor', 'Price', 'Maxed as % of price',
       f"vs {TG['worstCaseShareOfPrice']:.0%} target"], rows,
      'Per-tier cost at four utilisation levels, from the shipped caps')

table(['Metric'] + list(PLANS),
      [[m] + [usd(breakdown[t][m]) for t in PLANS] for m in breakdown['pro']],
      'Where each tier\'s worst case comes from')


**Per-tier cost at four utilisation levels, from the shipped caps**

| Tier | Lectures/mo | Light (25%) | Expected (50%) | Heavy (80%) | At caps | Price floor | Price | Maxed as % of price | vs 60% target |
|---|---|---|---|---|---|---|---|---|---|
| free | 4 | $1.45 | $2.89 | $4.63 | $5.79 | $6.32 | $0.00 | — | ✅ |
| fresh | 6 | $2.58 | $5.16 | $8.25 | $10.31 | $11.01 | $19.00 | 54% | ✅ |
| pro | 26 | $17.76 | $35.53 | $56.84 | $71.05 | $74.02 | $99.00 | 72% | ✅ |
| max | 40 | $43.43 | $86.86 | $138.98 | $173.73 | $180.53 | $299.00 | 58% | ✅ |


**Where each tier's worst case comes from**

| Metric | free | fresh | pro | max |
|---|---|---|---|---|
| aiTokens | $3.37 | $4.72 | $21.94 | $33.75 |
| sttMinutes | $0.00 | $0.00 | $9.60 | $52.80 |
| diarizationMinutes | $0.00 | $0.00 | $5.60 | $17.60 |
| ttsCharacters | $2.40 | $3.20 | $12.80 | $21.60 |
| ttsPremiumCharacters | $0.00 | $0.00 | $3.00 | $9.00 |
| aiImages | $0.00 | $0.00 | $3.36 | $10.08 |
| translationCharacters | $0.00 | $0.00 | $2.00 | $7.00 |
| audienceTtsCharacters | $0.00 | $1.28 | $7.20 | $11.20 |
| audienceLocales | $0.00 | $1.08 | $5.40 | $9.72 |
| audioStorageMb | $0.01 | $0.03 | $0.16 | $0.98 |


## 6. Break-even

Fixed costs are the same whatever we charge, which is why a cheap tier needs so
many more subscribers than an expensive one.


In [44]:
for scenario, costs in A['fixedCostsUsdPerMonth'].items():
    fixed = sum(costs.values())
    rows = [[k, usd(v)] for k, v in costs.items()] + [['**Total**', f'**{usd(fixed)}**']]
    table(['Line', 'Per month'], rows, f'Fixed costs — {scenario}')

    out = []
    for tier, plan in PLANS.items():
        price = A['tiers'][tier]['priceUsd']
        if not price:
            continue
        wc = sum(worst_case(plan['caps']).values())
        fee = PAY_RATE * price + PAY_FIXED
        subs = lambda c: '∞' if price - c - fee <= 0 else f'{-(-fixed // (price - c - fee)):.0f}'
        out.append([tier, usd(price), subs(wc * TG['lightUtilisation']),
                    subs(wc * TG['expectedUtilisation']),
                    subs(wc * TG['heavyUtilisation']), subs(wc)])
    table(['Tier', 'Price', 'Light use', 'Expected use', 'Heavy use', 'At caps'], out,
          f'Subscribers needed to cover {usd(fixed)}/month')


**Fixed costs — pilot**

| Line | Per month |
|---|---|
| appPlatformApi | $5.00 |
| spaces | $5.00 |
| atlasM10WithBackups | $66.94 |
| transactionalEmail | $0.00 |
| domain | $1.50 |
| developerTooling | $200.00 |
| **Total** | **$278.44** |


**Subscribers needed to cover $278.44/month**

| Tier | Price | Light use | Expected use | Heavy use | At caps |
|---|---|---|---|---|---|
| fresh | $19.00 | 19 | 22 | 29 | 37 |
| pro | $99.00 | 4 | 5 | 8 | 12 |
| max | $299.00 | 2 | 2 | 2 | 3 |


**Fixed costs — production**

| Line | Per month |
|---|---|
| appPlatformApi | $25.00 |
| spaces | $5.00 |
| atlasM10WithBackups | $66.94 |
| transactionalEmail | $15.00 |
| domain | $1.50 |
| developerTooling | $200.00 |
| **Total** | **$313.44** |


**Subscribers needed to cover $313.44/month**

| Tier | Price | Light use | Expected use | Heavy use | At caps |
|---|---|---|---|---|---|
| fresh | $19.00 | 21 | 25 | 33 | 41 |
| pro | $99.00 | 5 | 6 | 9 | 14 |
| max | $299.00 | 2 | 2 | 3 | 3 |


## 7. Sensitivity to lecture duration

The headline assumption. If real lectures are longer or shorter than 75 minutes,
this is what has to be re-decided.


In [45]:
rows = []
for d in A['sensitivityDurationsMinutes']:
    rr = per_lecture(d)
    t = sum(rr['lines'].values())
    pro = A['tiers']['pro']['lecturesPerMonth'] * rr['aiTokens']
    rows.append([f'{d} min', usd(t - rr['cloudOnly']), usd(t),
                 f'{pro/1e6:.0f}M', f"{rr['audioMb']:.0f} MB"])
table(['Duration', 'Per lecture (browser)', 'Per lecture (cloud)',
       'Pro aiTokens needed/mo', 'Audio per lecture'], rows,
      'Duration sensitivity')


**Duration sensitivity**

| Duration | Per lecture (browser) | Per lecture (cloud) | Pro aiTokens needed/mo | Audio per lecture |
|---|---|---|---|---|
| 50 min | $1.23 | $2.31 | 39M | 144 MB |
| 75 min | $1.81 | $3.42 | 59M | 216 MB |
| 110 min | $2.63 | $4.97 | 86M | 317 MB |


## 8. Caps the assumptions imply

Derives what each cap *would* be from the usage model, so you can see where the
shipped values were rounded up for headroom — or deliberately set below full
coverage, as `sttMinutes` and `diarizationMinutes` are, because at
$0.016/min full coverage would dominate the tier.


In [46]:
rows = []
for tier, cfg in A['tiers'].items():
    n = cfg['lecturesPerMonth']
    rr = per_lecture()
    derived = {
        'aiTokens': n * rr['aiTokens'],
        'ttsCharacters': n * (rr['narrationChars']
                              + A['revision']['resynthesizedSlides'] * A['lecture']['narrationCharsPerSlide']),
        'audienceLocales': n * A['audience']['localesPerDeck'],
        'imageLookups': n * rr['slides'] * (1 + A['revision']['enrichmentRedoShare']),
    }
    for metric, value in derived.items():
        shipped = PLANS[tier]['caps'].get(metric)
        rows.append([tier, metric, f'{value:,.0f}',
                     'unlimited' if shipped is None else f'{shipped:,}',
                     '—' if shipped in (None, 0) else f'{shipped/value:.2f}x'])
table(['Tier', 'Metric', 'Implied by assumptions', 'Shipped cap', 'Headroom'], rows,
      'Derived vs shipped')


**Derived vs shipped**

| Tier | Metric | Implied by assumptions | Shipped cap | Headroom |
|---|---|---|---|---|
| free | aiTokens | 9,045,000 | 10,000,000 | 1.11x |
| free | ttsCharacters | 120,000 | 150,000 | 1.25x |
| free | audienceLocales | 2 | 0 | — |
| free | imageLookups | 198 | 250 | 1.26x |
| fresh | aiTokens | 13,567,500 | 14,000,000 | 1.03x |
| fresh | ttsCharacters | 180,000 | 200,000 | 1.11x |
| fresh | audienceLocales | 3 | 3 | 1.00x |
| fresh | imageLookups | 297 | 350 | 1.18x |
| pro | aiTokens | 58,792,500 | 65,000,000 | 1.11x |
| pro | ttsCharacters | 780,000 | 800,000 | 1.03x |
| pro | audienceLocales | 13 | 15 | 1.15x |
| pro | imageLookups | 1,287 | 1,500 | 1.17x |
| max | aiTokens | 90,450,000 | 100,000,000 | 1.11x |
| max | ttsCharacters | 1,200,000 | 1,350,000 | 1.12x |
| max | audienceLocales | 20 | 27 | 1.35x |
| max | imageLookups | 1,980 | 2,200 | 1.11x |


## 9. Fiddling with this

- **Lecture length wrong?** `assumptions.lecture.durationMinutes` — everything rescales.
- **Students review more than we think?** `assumptions.audience.*`. Note playback of
  cached content is nearly free; only *new* languages spend, so `localesPerDeck` moves
  the number far more than `playbacksPerViewingStudent` does.
- **Vendor changed a price?** `pricing.json`, then re-run — the drift check in §1 will
  tell you it no longer matches what the server bills against.
- **Trying a different model?** `pricing.ai.defaultModel`. 2.5 Flash-Lite is ~2.5×
  cheaper than the current default; 3.5 Flash is ~6× dearer.
- **Considering a cap change?** Edit `../config/plans.json` and re-run §5 and §6 to see
  the effect on worst-case cost and break-even before shipping it.

When a change here should stick, update
[`docs/BILLING_COST_MODEL.md`](../docs/BILLING_COST_MODEL.md) so the prose and the
notebook agree — the doc is the explanation, this is the calculator.
